In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 25. 量子化

> **学習目標**
> - FP32, FP16, BF16, INT8, INT4 精度··速度
> - $q = \mathrm{round}(x/s) + z$ 複雑度
> - GPTQ, AWQ, GGUF

## 25.1

LLM · :
- LLaMA-7B FP16: 14GB
- 推論 KV Cache
- /

 :
- FP16 → INT8: , 速度 2
- FP16 → INT4: 1/4, 速度 3-4

## 25.2

| | | | | |
|---|---|---|---|---|
| FP32 | 1 | 8 | 23 | 32 bit |
| FP16 | 1 | 5 | 10 | 16 bit |
| BF16 | 1 | 8 | 7 | 16 bit |
| FP8 (E4M3) | 1 | 4 | 3 | 8 bit |

- FP16: 精度 ( )
- BF16: FP32 , 精度 → LLM 学習


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

#   比較
print("点  :")
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16), ('BF16', torch.bfloat16)]:
    info = torch.finfo(dtype)
    print(f"  {name}: min={info.min:.2e}, max={info.max:.2e}, eps={info.eps:.2e}")

# FP16  
print("\nFP16 :")
print(f"  2^15 = {float(torch.tensor(2**15, dtype=torch.float16))}")
print(f"  2^16 = {float(torch.tensor(2**16, dtype=torch.float16))}  # inf!")
print(f"  BF16 2^16 = {float(torch.tensor(2**16, dtype=torch.bfloat16))}  # ")


## 25.3

FP32 値 INT8 (0~255 -128~127) :

** ** (zero-point = 0):
$$q = \mathrm{round}(x / s), \quad s = \frac{\max|x|}{127}$$
$$\hat{x} = s \cdot q$$

** ** (zero-point ):
$$q = \mathrm{round}(x / s) + z$$
$$s = \frac{\max(x) - \min(x)}{255}, \quad z = -\mathrm{round}(\min(x) / s)$$

 誤差: $\epsilon = x - \hat{x}$.


In [ ]:
#  
def quantize_symmetric(x, n_bits=8):
    """Symmetric Quantization."""
    qmax = 2 ** (n_bits - 1) - 1  # 127 for INT8
    scale = x.abs().max() / qmax
    q = torch.round(x / scale).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.to(torch.float32) * scale

# Test
torch.manual_seed(0)
x = torch.randn(100) * 5  # Mean 0, Standard Deviation 5
q, scale = quantize_symmetric(x, n_bits=8)
x_hat = dequantize_symmetric(q, scale)
error = (x - x_hat).abs()
print(f" : [{x.min():.3f}, {x.max():.3f}]")
print(f"Quantization : [{q.min()}, {q.max()}]")
print(f": {scale:.6f}")
print(f"Mean Absolute error: {error.mean():.6f}")
print(f"Maximum Absolute error: {error.max():.6f}")
print(f" Error: {(error.mean() / x.abs().mean() * 100):.3f}%")

#    比較
print("\n  Quantization Error:")
for n_bits in [8, 4, 2]:
    q, scale = quantize_symmetric(x, n_bits=n_bits)
    x_hat = dequantize_symmetric(q, scale)
    err = (x - x_hat).abs().mean()
    print(f"  INT{n_bits}: Mean Error = {err:.6f} ({err/x.abs().mean()*100:.2f}%)")


## 25.4 Per-tensor, Per-channel, Per-group

- **Per-tensor**: scale. 複雑度 .
- **Per-channel**: (/) scale. 複雑度 , .
- **Per-group**: $g$ scale. . LLM .

: 4-bit group_size=128 → 128 値 scale (FP16) .


In [ ]:
# Per-group 
def quantize_per_group(x, n_bits=4, group_size=128):
    """ Quantization."""
    qmax = 2 ** (n_bits - 1) - 1
    #  
    orig_shape = x.shape
    x_flat = x.reshape(-1, group_size)
    scales = x_flat.abs().max(dim=1, keepdim=True).values / qmax
    q = torch.round(x_flat / scales).clamp(-qmax - 1, qmax)
    return q, scales, orig_shape

def dequantize_per_group(q, scales, orig_shape):
    return (q * scales).reshape(orig_shape)

# Comparison: per-tensor vs per-group
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.1  # Weight Matrix

# per-tensor (4-bit)
qmax = 7  # 4-bit symmetric
scale_t = W.abs().max() / qmax
q_t = torch.round(W / scale_t).clamp(-8, 7)
W_hat_t = q_t * scale_t
err_t = (W - W_hat_t).abs().mean()

# per-group (4-bit, group=128)
q_g, scales_g, shape = quantize_per_group(W, n_bits=4, group_size=128)
W_hat_g = dequantize_per_group(q_g, scales_g, shape)
err_g = (W - W_hat_g).abs().mean()

print(f"4-bit  誤差:")
print(f"  Per-tensor: {err_t:.6f} ({err_t/W.abs().mean()*100:.2f}%)")
print(f"  Per-group (128): {err_g:.6f} ({err_g/W.abs().mean()*100:.2f}%)")
print("\n=> Per-group  . LLM Quantization Standard.")


## 25.5 PTQ (Post-Training Quantization) vs QAT

**PTQ**: 学習 . 複雑度 .

**QAT (Quantization-Aware Training)**: 学習. .

LLM PTQ (学習 ).

## 25.6 GPTQ, AWQ

### GPTQ
- 2 誤差
- ( )
- Hessian 行列 誤差
- INT4複雑度 PPL

### AWQ (Activation-aware Weight Quantization)
- ( ) FP16
- 値 複雑度
- GPTQ


## 25.7 QLoRA — 4-bit + LoRA

QLoRA (Dettmers et al., 2023):
1. モデル 4-bit NF4 
2. LoRA FP16/BF16 学習
3. : 7B モデル 24GB GPU

NF4 (NormalFloat 4-bit): 分布 4-bit .


In [ ]:
# QLoRA  
def qlora_memory(base_params_b, adapter_ratio=0.01):
    """QLoRA Memory Estimation (GB).
    base_params_b:  Parameter Count (: 10)
    adapter_ratio:   Ratio
    """
    # : 4-bit = 0.5 bytes
    base_mem = base_params_b * 0.5
    # : FP16 + AdamW (4) = 8 bytes
    adapter_mem = base_params_b * adapter_ratio * 8
    # 値 ()
    activation = 2  # GB
    return base_mem + adapter_mem + activation

# Comparison:   vs QLoRA
print(f"{'Model':>10} | {' FT FP16 (GB)':>15} | {'QLoRA 4-bit (GB)':>17}")
print("-" * 50)
for name, n_b in [('7B', 7), ('13B', 13), ('30B', 30), ('70B', 70)]:
    full = n_b * 2 * 4 + 4  # FP16 + AdamW FP32 (4) + 
    qlora = qlora_memory(n_b, 0.01)
    print(f"{name:>10} | {full:>15.1f} | {qlora:>17.1f}")
print("\n=> QLoRA 70B Model度  48GB GPU  .")


## 25.8 [CPU/GPU ベンチマーク ⑪] 精度 推論


In [ ]:
# 精度 推論 速度 比較
from llm_math.bench import time_fn

#   推論
def bench_linear(d_in, d_out, batch_size, dtype, device='cpu'):
    """LinearLayer Inference Time."""
    model = nn.Linear(d_in, d_out).to(dtype=dtype, device=device)
    x = torch.randn(batch_size, d_in, dtype=dtype, device=device)
    def run():
        with torch.no_grad():
            return model(x)
    return run

print(f"{'dtype':>10} | {'CPU (ms)':>10} | {'Memory (MB)':>12}")
print("-" * 40)
d_in, d_out, bs = 4096, 4096, 8
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16)]:
    if dtype == torch.float16 and not torch.cuda.is_available():
        # CPU FP16 
        continue
    try:
        run = bench_linear(d_in, d_out, bs, dtype, 'cpu')
        # warmup
        run()
        res = time_fn(run, device='cpu', warmup=2, repeat=5)
        # Memory ()
        mem = d_in * d_out * (4 if dtype == torch.float32 else 2) / 1024**2
        print(f"{name:>10} | {res['mean_ms']:>10.3f} | {mem:>12.2f}")
    except Exception as e:
        print(f"{name:>10} | {'N/A':>10} | error: {e}")

print("\n=> FP16 Memory , CPU Speed Improvement  (GPU  Improvement).")


## 25.9 要点

| | | | 複雑度 |
|---|---|---|---|
| FP16 | 16 | 2x | |
| BF16 | 16 | 2x | |
| INT8 | 8 | 4x | |
| INT4 (GPTQ/AWQ) | 4 | 8x | |
| QLoRA (NF4) | 4 | 8x | |

## 演習問題

1. INT8 出力 MSE .
2. Per-tensor vs Per-channel vs Per-group 誤差 比較.
3. 32, 64, 128, 256 誤差 比較.
4. FP32 vs FP16 推論 速度 CPU GPU 比較.
5. QLoRA 比較 7B, 13B, 70B モデル 計算.

> 解答: `solutions/ch25_solutions.ipynb`
